# Professional Digital Twin — Project Development Notebook

This notebook documents the development of my first independent Agentic AI application: a professional Digital Twin.

Unlike the Foundations notebook, which records how I learned the underlying concepts, this notebook focuses on the **actual project**: its architecture, implementation, tools, Agent Loop, interface, deployment, testing, and ongoing improvements.

**Live Demo:** https://huggingface.co/spaces/Sollie/twin

**Status:** Active Development

## 1. Project Overview

The Digital Twin is a conversational AI application that represents my professional background, skills, experience, and career journey. It is designed for visitors such as potential clients, collaborators, or future employers.

The current implementation combines:
- professional context from `linkedin.pdf` and `summary.txt`
- an OpenAI-powered conversational model
- conversation history
- function/tool calling
- a no-framework Agent Loop
- Pushover notifications
- a Gradio interface
- modular Python files
- Hugging Face deployment

## 2. Project Structure

```text
professional-digital-twin/
├── README.md
├── professional-digital-twin.ipynb
├── app.py
├── context.py
├── tools.py
├── styles.py
├── requirements.txt
├── linkedin.pdf
└── summary.txt
```

The notebook is a development record. The Python modules are the application itself.

## 3. Application Setup

The application uses Python to connect the user interface, professional context, OpenAI model, and external tools. Sensitive credentials are loaded from environment variables rather than being written into the source code.

In [ ]:
# 18. Expand the Digital Twin with practical tools
# imports

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr


In [ ]:

load_dotenv(override=True)
openai = OpenAI()


## 4. Professional Context

The Digital Twin needs reliable information about the person it represents. The current project uses two source files:

- `linkedin.pdf` — professional profile and experience
- `summary.txt` — additional personal/professional context

This is the project's context-engineering layer.

In [ ]:
# 27. Reload the Digital Twin's source context
reader = PdfReader("linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()


## 5. Digital Twin Behavior

The system prompt defines the Digital Twin's identity, professional scope, behavior, and rules for handling unknown information.

The current design instructs the agent to stay focused on career, background, skills, and experience; avoid inventing information; record unanswered questions; and capture contact interest when appropriate.

In [ ]:
# 28. Refine the Digital Twin's operating rules
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Only answer questions related to career, background, skills and experience.
If the user asks about something unrelated, then steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

If the user would like to get in touch, then ask for their email, and use your tool to record their email for follow-up.

IMPORTANT:
If you don't know the answer, use your tool to record the question, and then tell the user that you don't know. Never make up an answer.
"""


## 6. Tooling and External Actions

The Digital Twin can move beyond generating text by calling Python tools. The current project includes Pushover-based external notification functionality and tools for recording contact interest and unanswered questions.

In [ ]:
# 19. Load Pushover configuration

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")


In [ ]:
# 20. Create a notification function
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)


In [ ]:
# 21. Tool for recording contact interest
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return "OK"


In [ ]:
# 22. Tool for unanswered questions
def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return "OK"


### Tool schemas

The Python functions define what the tools actually do. The JSON schemas describe those tools to the model so it can decide when to request them.

In [ ]:
# 23. Describe the contact tool to the LLM
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"},
            "name": {"type": "string", "description": "The user's name, if they provided it"},
            "notes": {"type": "string", "description": "Any additional info about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}


In [ ]:
# Describe the unknown-question tool
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The question that couldn't be answered"},
        },
        "required": ["question"],
        "additionalProperties": False
    }
}


In [ ]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]


## 7. Dynamic Tool Execution

The first implementation used explicit `if/elif` mapping between tool names and Python functions. The project then moved to dynamic lookup with `globals()`.

The resulting flow is:

**LLM chooses a tool → Python finds the function → JSON arguments become Python arguments → tool executes → result returns to the LLM**

In [ ]:
# 24. First approach: manually map tool names to functions

def handle_tool_calls_with_manual_if(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        # THE BIG IF STATEMENT!!!

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)

        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results


In [ ]:
# 25. Understand `globals()`
globals()["record_unknown_question"]("this is a really hard question")


In [ ]:
# 26. Dynamically execute the requested tool

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else "No tool found"
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results


## 8. Agent Loop

The Digital Twin becomes an agent when it can request a tool, receive the tool result, and continue the conversation rather than stopping after the first model response.

```text
User message
     ↓
OpenAI model
     ↓
Tool call? ── No ──→ Final response
     │
    Yes
     ↓
Python tool
     ↓
Tool result
     ↓
OpenAI model
     ↓
Final response
```

In [ ]:
# 29. Build the complete Lab 4 Agent Loop
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    return response.choices[0].message.content


## 9. Gradio Interface

Gradio provides the public conversational interface around the `chat()` function.

In [ ]:
# 30. Put the complete Digital Twin behind Gradio
gr.ChatInterface(chat).launch(inbrowser=True)


## 10. From Notebook to Application

The project was separated into reusable Python modules so the Digital Twin could run independently of Jupyter.

- `app.py` — application and chat interface
- `context.py` — professional/personal context
- `tools.py` — tool functions
- `styles.py` — interface styling
- `linkedin.pdf` and `summary.txt` — source context
- `requirements.txt` — dependencies

The local application can be started with:

```bash
uv run app.py
```

## 11. Deployment

The application is publicly deployed as a Gradio Space on Hugging Face.

**Live Demo:** https://huggingface.co/spaces/Sollie/twin

Deployment secrets are kept outside the repository:

```text
OPENAI_API_KEY
PUSHOVER_USER
PUSHOVER_TOKEN
```

The application code and deployment configuration are kept separate from the secret values.

## 12. Testing and Current Issues

The deployed application is currently treated as an active development project rather than a finished production system.

During testing, I identified an issue with the intended out-of-scope/notification behavior: a question outside the Digital Twin's professional scope did not produce the expected Pushover notification. This is now a debugging task rather than something to hide from the project record.

Other planned improvements include:
- improving tool reliability
- strengthening scope control
- adding a work-focused evaluator
- improving Agent Loop behavior
- expanding useful tools
- improving context and retrieval
- exploring memory and higher-level agent frameworks

## 13. What This Project Represents

This project is the transition from learning individual Agentic AI concepts to combining them into a deployed application.

The development path is:

**Concepts → implementation → tools → Agent Loop → modular application → deployment → testing → iteration**

The Foundations notebook documents how I learned these mechanics. This notebook documents how those mechanics became an actual project.